# Customer Churn Prediction

**Goal:** predict which telecom customers will cancel, and find out what drives it.

**Data:** Telco Customer Churn (Kaggle), about 7,000 customers.

**Plan:** explore → clean → baseline → compare models → tune → explain → business takeaways.

In [2]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score, f1_score,
                             ConfusionMatrixDisplay, RocCurveDisplay, precision_recall_curve)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")                                                       
sns.set_theme(style="whitegrid")
os.makedirs("images", exist_ok=True)
RANDOM_STATE = 42

ImportError: DLL load failed while importing _barnes_hut_tsne: An Application Control policy has blocked this file.

## 1. Load the data

In [ ]:
# Change this path to wherever your CSV is saved
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isna().sum()[df.isna().sum() > 0])

## 2. Clean the data

`TotalCharges` is stored as text because some rows contain a blank space. Those rows are brand-new customers (tenure = 0), so a total of 0 makes sense.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Rows with missing TotalCharges:", df["TotalCharges"].isna().sum())
print("Their tenure values:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())

df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
df.duplicated().sum()

## 3. Exploratory analysis

In [ ]:
churn_rate = df["Churn"].mean()
print(f"Overall churn rate: {churn_rate:.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
df["Churn"].map({0: "Stayed", 1: "Churned"}).value_counts().plot.bar(ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_title("Customers who stayed vs churned")
ax.set_ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/01_churn_balance.png", dpi=150)
plt.show()

The classes are imbalanced (about 3 in 4 customers stay). A model that always predicts 'stayed' would score about 73% accuracy and still be useless, so I will judge models with **ROC-AUC, recall and F1** instead of accuracy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

(df.groupby("Contract")["Churn"].mean() * 100).plot.bar(ax=axes[0], color="#DD8452")
axes[0].set_title("Churn rate by contract type (%)")
axes[0].set_ylabel("%")
axes[0].tick_params(axis="x", rotation=0)

sns.histplot(data=df, x="tenure", hue="Churn", bins=30, ax=axes[1], multiple="stack")
axes[1].set_title("Tenure (months) by churn")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[2])
axes[2].set_title("Monthly charges by churn")
axes[2].set_xticklabels(["Stayed", "Churned"])

plt.tight_layout()
plt.savefig("images/02_eda_drivers.png", dpi=150)
plt.show()

In [ ]:
for col in ["InternetService", "PaymentMethod", "TechSupport", "PaperlessBilling"]:
    print((df.groupby(col)["Churn"].mean() * 100).round(1).sort_values(ascending=False), "\n")

**What I noticed:** _(write 3 bullet points in your own words after looking at the charts, e.g. which contract type churns most, whether new customers leave early, what internet service type stands out)_

## 4. Train/test split and preprocessing

In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X.select_dtypes(include="object").columns.tolist()

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="passthrough")   # SeniorCitizen is already 0/1

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train/test:", y_train.mean().round(3), y_test.mean().round(3))

## 5. Compare models with cross-validation

Logistic regression is the **baseline**. The other two models have to beat it to justify their extra complexity.

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced_subsample",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                             scale_pos_weight=neg / pos, eval_metric="logloss",
                             random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    rows.append({"Model": name, "CV ROC-AUC": auc.mean(), "CV F1": f1.mean()})

cv_results = pd.DataFrame(rows).set_index("Model").round(3)
cv_results

## 6. Tune the best candidate (XGBoost)

In [ ]:
xgb_pipe = Pipeline([("prep", preprocess),
                     ("model", XGBClassifier(scale_pos_weight=neg / pos, eval_metric="logloss",
                                             random_state=RANDOM_STATE))])

param_dist = {
    "model__n_estimators": [150, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
}

search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=25, scoring="roc_auc",
                            cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
print("Best CV ROC-AUC:", round(search.best_score_, 3))
print("Best params:", search.best_params_)
best_model = search.best_estimator_

## 7. Final evaluation on the held-out test set

In [ ]:
# Baseline on the test set for comparison
baseline = Pipeline([("prep", preprocess), ("model", models["Logistic Regression"])]).fit(X_train, y_train)

for name, m in [("Logistic Regression (baseline)", baseline), ("Tuned XGBoost", best_model)]:
    proba = m.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"{name}: ROC-AUC = {roc_auc_score(y_test, proba):.3f} | F1 = {f1_score(y_test, pred):.3f}")

proba = best_model.predict_proba(X_test)[:, 1]
print("\n", classification_report(y_test, (proba >= 0.5).astype(int), target_names=["Stayed", "Churned"]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, (proba >= 0.5).astype(int),
                                        display_labels=["Stayed", "Churned"], ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix (threshold 0.5)")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1], name="Tuned XGBoost")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.savefig("images/03_model_evaluation.png", dpi=150)
plt.show()

## 8. Choose a threshold for the business

Missing a churner (lost customer) usually costs more than sending a retention offer to someone who would have stayed. So the default 0.5 threshold may not be the best choice.

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, proba)
plt.figure(figsize=(7, 4))
plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.xlabel("Decision threshold"); plt.legend(); plt.title("Precision vs recall by threshold")
plt.tight_layout()
plt.savefig("images/04_threshold.png", dpi=150)
plt.show()

for t in [0.3, 0.4, 0.5, 0.6]:
    pred = (proba >= t).astype(int)
    print(f"threshold {t}: flagged {pred.sum():4d} customers | "
          f"recall {(pred[y_test == 1]).mean():.2f} | precision {y_test[pred == 1].mean():.2f}")

**My choice:** _(pick a threshold and explain in one or two sentences, e.g. 'I would use 0.4 so we catch about X% of churners while still contacting a manageable number of customers')_

## 9. What drives churn? (SHAP)

In [ ]:
prep = best_model.named_steps["prep"]
xgb = best_model.named_steps["model"]

X_test_t = pd.DataFrame(prep.transform(X_test), columns=prep.get_feature_names_out())
X_test_t.columns = [c.split("__", 1)[-1] for c in X_test_t.columns]

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test_t)

shap.summary_plot(shap_values, X_test_t, max_display=12, show=False)
plt.tight_layout()
plt.savefig("images/05_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

**Top drivers:** _(read the plot: features at the top matter most. Red dots on the right mean a high value pushes towards churn. Write the top 3 in plain English.)_

## 10. Conclusions and recommendations

- **Model:** tuned XGBoost reached ROC-AUC of **___** and F1 of **___** on unseen data, versus **___** / **___** for the logistic regression baseline.
- **Main drivers of churn:** ___, ___, ___
- **Recommendation:** ___ (for example: offer discounts to move month-to-month customers onto longer contracts)
- **Limitations:** one dataset from one company, no time dimension, no cost data for retention offers.
- **Next steps:** estimate the money saved by a retention campaign, and test the model on newer data.